In [2]:
!pip install python-dotenv

In [12]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import json
from dotenv import load_dotenv
import os

In [13]:
load_dotenv()

True

In [14]:
SERVICE_KEY = os.getenv('SERVICE_KEY')

# API 기본 URL들

In [15]:
BASE_URLS = {
    "단기예보": "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst",
    "초단기예보": "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtFcst", 
    "초단기실황": "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst",
    "기상특보": "http://apis.data.go.kr/1360000/WthrWrnInfoService/getWthrWrnList",
    "지상일자료": "http://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList"
}

## 🏛️ 3단계: 서울 25개구 좌표 정보

In [16]:
seoul_districts = {
    "종로구": {"nx": 60, "ny": 127},
    "중구": {"nx": 60, "ny": 127},
    "용산구": {"nx": 60, "ny": 126},
    "성동구": {"nx": 61, "ny": 127},
    "광진구": {"nx": 62, "ny": 126},
    "동대문구": {"nx": 61, "ny": 127},
    "중랑구": {"nx": 62, "ny": 128},
    "성북구": {"nx": 61, "ny": 127},
    "강북구": {"nx": 61, "ny": 128},
    "도봉구": {"nx": 61, "ny": 129},
    "노원구": {"nx": 61, "ny": 129},
    "은평구": {"nx": 59, "ny": 127},
    "서대문구": {"nx": 59, "ny": 127},
    "마포구": {"nx": 59, "ny": 126},
    "양천구": {"nx": 58, "ny": 126},
    "강서구": {"nx": 58, "ny": 126},
    "구로구": {"nx": 58, "ny": 125},
    "금천구": {"nx": 59, "ny": 124},
    "영등포구": {"nx": 58, "ny": 126},
    "동작구": {"nx": 59, "ny": 125},
    "관악구": {"nx": 59, "ny": 125},
    "서초구": {"nx": 60, "ny": 125},
    "강남구": {"nx": 61, "ny": 125},
    "송파구": {"nx": 62, "ny": 125},
    "강동구": {"nx": 62, "ny": 126}
}

## 📊 4단계: 단기예보 데이터 수집 함수

In [17]:
def get_weather_forecast(district_name, base_date=None, base_time="0500"):
    """
    특정 구의 단기예보 데이터를 가져오는 함수
    
    Args:
        district_name: 구 이름 (예: "강남구")
        base_date: 예보 기준일 (YYYYMMDD 형식, None이면 오늘)
        base_time: 예보 기준시간 (0200, 0500, 0800, 1100, 1400, 1700, 2000, 2300)
    
    Returns:
        DataFrame: 기상 예보 데이터
    """
    if base_date is None:
        base_date = datetime.now().strftime("%Y%m%d")
    
    if district_name not in seoul_districts:
        print(f"❌ {district_name}는 서울 25개구에 포함되지 않습니다.")
        return None
    
    nx = seoul_districts[district_name]["nx"]
    ny = seoul_districts[district_name]["ny"]
    
    params = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": 1000,
        "pageNo": 1,
        "dataType": "JSON",
        "base_date": base_date,
        "base_time": base_time,
        "nx": nx,
        "ny": ny
    }
    
    try:
        response = requests.get(BASE_URLS["단기예보"], params=params)
        response.raise_for_status()
        
        data = response.json()
        
        if data["response"]["header"]["resultCode"] == "00":
            items = data["response"]["body"]["items"]["item"]
            df = pd.DataFrame(items)
            df["district"] = district_name
            return df
        else:
            print(f"❌ API 오류: {data['response']['header']['resultMsg']}")
            return None
            
    except requests.exceptions.RequestException as e:
        print(f"❌ 네트워크 오류: {e}")
        return None
    except Exception as e:
        print(f"❌ 기타 오류: {e}")
        return None


# 사용 예시

In [18]:
gangnam_weather = get_weather_forecast("강남구")
if gangnam_weather is not None:
    print("✅ 강남구 날씨 데이터 수집 완료!")
    print(gangnam_weather.head())

❌ 네트워크 오류: Expecting value: line 1 column 1 (char 0)


## 🌡️ 5단계: 모든 구의 데이터 수집

In [19]:
def collect_all_seoul_weather(base_date=None, base_time="0500"):
    """서울 전체 25개구의 기상 데이터를 수집"""
    all_data = []
    
    print("🌤️ 서울 25개구 기상 데이터 수집 시작...")
    
    for i, district in enumerate(seoul_districts.keys(), 1):
        print(f"[{i:2d}/25] {district} 데이터 수집 중...")
        
        weather_data = get_weather_forecast(district, base_date, base_time)
        
        if weather_data is not None:
            all_data.append(weather_data)
            print(f"    ✅ {district} 완료 (데이터 {len(weather_data)}개)")
        else:
            print(f"    ❌ {district} 실패")
    
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        print(f"\n🎉 전체 수집 완료! 총 {len(combined_df)}개 데이터")
        return combined_df
    else:
        print("❌ 수집된 데이터가 없습니다.")
        return None


# 전체 데이터 수집

In [20]:
seoul_weather_data = collect_all_seoul_weather()

🌤️ 서울 25개구 기상 데이터 수집 시작...
[ 1/25] 종로구 데이터 수집 중...
❌ 네트워크 오류: Expecting value: line 1 column 1 (char 0)
    ❌ 종로구 실패
[ 2/25] 중구 데이터 수집 중...
❌ 네트워크 오류: Expecting value: line 1 column 1 (char 0)
    ❌ 중구 실패
[ 3/25] 용산구 데이터 수집 중...
❌ 네트워크 오류: Expecting value: line 1 column 1 (char 0)
    ❌ 용산구 실패
[ 4/25] 성동구 데이터 수집 중...
❌ 네트워크 오류: Expecting value: line 1 column 1 (char 0)
    ❌ 성동구 실패
[ 5/25] 광진구 데이터 수집 중...
❌ 네트워크 오류: Expecting value: line 1 column 1 (char 0)
    ❌ 광진구 실패
[ 6/25] 동대문구 데이터 수집 중...
❌ 네트워크 오류: Expecting value: line 1 column 1 (char 0)
    ❌ 동대문구 실패
[ 7/25] 중랑구 데이터 수집 중...
❌ 네트워크 오류: Expecting value: line 1 column 1 (char 0)
    ❌ 중랑구 실패
[ 8/25] 성북구 데이터 수집 중...
❌ 네트워크 오류: Expecting value: line 1 column 1 (char 0)
    ❌ 성북구 실패
[ 9/25] 강북구 데이터 수집 중...
❌ 네트워크 오류: Expecting value: line 1 column 1 (char 0)
    ❌ 강북구 실패
[10/25] 도봉구 데이터 수집 중...
❌ 네트워크 오류: Expecting value: line 1 column 1 (char 0)
    ❌ 도봉구 실패
[11/25] 노원구 데이터 수집 중...
❌ 네트워크 오류: Expecting value: line 1 column 1 (char

## 🔧 6단계: 데이터 전처리

In [21]:
def preprocess_weather_data(df):
    """기상 데이터 전처리"""
    if df is None or df.empty:
        return None
    
    # 날짜시간 컬럼 생성
    df['datetime'] = pd.to_datetime(df['fcstDate'] + df['fcstTime'], format='%Y%m%d%H%M')
    
    # 기상 요소별로 데이터 피벗
    pivot_df = df.pivot_table(
        index=['district', 'datetime'],
        columns='category',
        values='fcstValue',
        aggfunc='first'
    ).reset_index()
    
    # 컬럼명 정리
    pivot_df.columns.name = None
    
    # 숫자형 변환 (가능한 컬럼들)
    numeric_columns = ['TMP', 'UUU', 'VVV', 'VEC', 'WSD', 'SKY', 'PTY', 'POP', 'WAV', 'PCP', 'REH', 'SNO']
    for col in numeric_columns:
        if col in pivot_df.columns:
            pivot_df[col] = pd.to_numeric(pivot_df[col], errors='coerce')
    
    # 기상 코드 해석
    if 'SKY' in pivot_df.columns:
        pivot_df['sky_status'] = pivot_df['SKY'].map({1: '맑음', 3: '구름많음', 4: '흐림'})
    
    if 'PTY' in pivot_df.columns:
        pivot_df['precipitation_type'] = pivot_df['PTY'].map({
            0: '강수없음', 1: '비', 2: '비/눈', 3: '눈', 5: '빗방울', 6: '빗방울눈날림', 7: '눈날림'
        })
    
    return pivot_df

# 데이터 전처리 실행

In [22]:
if seoul_weather_data is not None:
    processed_data = preprocess_weather_data(seoul_weather_data)
    print("✅ 데이터 전처리 완료!")
    print(f"📊 처리된 데이터 형태: {processed_data.shape}")
    print("\n📈 컬럼 정보:")
    print(processed_data.columns.tolist())

## 📈 7단계: 데이터 시각화

In [23]:
def visualize_seoul_weather(df):
    """서울 구별 기상 데이터 시각화"""
    if df is None or df.empty:
        return
    
    plt.style.use('default')
    plt.rcParams['font.family'] = 'DejaVu Sans'
    plt.rcParams['axes.unicode_minus'] = False
    
    # 1. 구별 평균 기온 비교
    if 'TMP' in df.columns:
        plt.figure(figsize=(15, 8))
        avg_temp_by_district = df.groupby('district')['TMP'].mean().sort_values(ascending=False)
        
        plt.subplot(2, 2, 1)
        avg_temp_by_district.plot(kind='bar', color='skyblue', alpha=0.7)
        plt.title('Seoul Districts - Average Temperature')
        plt.xlabel('District')
        plt.ylabel('Temperature (°C)')
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
    
    # 2. 시간별 기온 변화
    if 'TMP' in df.columns:
        plt.subplot(2, 2, 2)
        # 상위 5개구만 표시
        top_districts = avg_temp_by_district.head(5).index
        for district in top_districts:
            district_data = df[df['district'] == district].set_index('datetime')['TMP']
            plt.plot(district_data.index, district_data.values, marker='o', label=district, linewidth=2)
        
        plt.title('Temperature Change Over Time (Top 5 Districts)')
        plt.xlabel('Time')
        plt.ylabel('Temperature (°C)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
    
    # 3. 습도 분포
    if 'REH' in df.columns:
        plt.subplot(2, 2, 3)
        df.boxplot(column='REH', by='district', ax=plt.gca())
        plt.title('Humidity Distribution by District')
        plt.xlabel('District')
        plt.ylabel('Humidity (%)')
        plt.xticks(rotation=45)
    
    # 4. 강수확률 히트맵
    if 'POP' in df.columns:
        plt.subplot(2, 2, 4)
        pop_pivot = df.pivot_table(
            index='district',
            columns=df['datetime'].dt.hour,
            values='POP',
            aggfunc='mean'
        )
        sns.heatmap(pop_pivot, cmap='Blues', annot=True, fmt='.1f', cbar_kws={'label': 'Precipitation Probability (%)'})
        plt.title('Precipitation Probability Heatmap')
        plt.xlabel('Hour')
        plt.ylabel('District')
    
    plt.tight_layout()
    plt.show()

# 시각화 실행

In [24]:
if 'processed_data' in locals() and processed_data is not None:
    visualize_seoul_weather(processed_data)

## 💾 8단계: 데이터 저장 및 활용

In [ ]:
def save_weather_data(df, filename_prefix="seoul_weather"):
    """기상 데이터를 다양한 형식으로 저장"""
    if df is None or df.empty:
        return
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # CSV 저장
    csv_filename = f"{filename_prefix}_{timestamp}.csv"
    df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
    print(f"📁 CSV 파일 저장: {csv_filename}")
    
    # Excel 저장
    excel_filename = f"{filename_prefix}_{timestamp}.xlsx"
    with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Weather_Data', index=False)
        
        # 구별 요약 통계 추가
        if 'TMP' in df.columns:
            summary = df.groupby('district').agg({
                'TMP': ['mean', 'min', 'max'],
                'REH': ['mean'],
                'POP': ['mean']
            }).round(2)
            summary.to_excel(writer, sheet_name='District_Summary')
    
    print(f"📊 Excel 파일 저장: {excel_filename}")
    
    return csv_filename, excel_filename

# 데이터 저장

In [ ]:
if 'processed_data' in locals() and processed_data is not None:
    csv_file, excel_file = save_weather_data(processed_data)

## 🔄 9단계: 정기적 데이터 수집 (선택사항)

In [ ]:
def setup_scheduled_collection():
    """정기적으로 데이터를 수집하는 함수"""
    import schedule
    import time
    
    def job():
        print(f"🕐 {datetime.now()} - 정기 데이터 수집 시작")
        data = collect_all_seoul_weather()
        if data is not None:
            processed = preprocess_weather_data(data)
            if processed is not None:
                save_weather_data(processed, "scheduled_seoul_weather")
                print("✅ 정기 데이터 수집 완료!")
    
    # 매일 오전 6시에 실행
    schedule.every().day.at("06:00").do(job)
    
    print("⏰ 정기 데이터 수집이 설정되었습니다 (매일 오전 6시)")
    print("⚠️  실행을 유지하려면 아래 루프를 실행하세요:")
    print("""
    while True:
        schedule.run_pending()
        time.sleep(60)  # 1분마다 체크
    """)

# setup_scheduled_collection()  # 필요시 주석 해제

# 10단계: 추가 분석 아이디어

# 10단계: 추가 분석 아이디어
"""
1. 머신러닝 모델링:
   - 과거 데이터로 기온 예측 모델 구축
   - 구별 날씨 패턴 클러스터링

2. 고급 시각화:
   - Folium으로 서울 지도에 기상 데이터 표시
   - Plotly로 인터랙티브 대시보드 구축

3. 데이터베이스 연동:
   - SQLite/MySQL에 데이터 저장
   - 시계열 데이터 분석

4. 웹 애플리케이션:
   - Flask/Django로 실시간 날씨 웹앱 구축
   - API 서버 구축

5. 알림 시스템:
   - 특정 조건 만족시 이메일/SMS 알림
   - 기상특보 자동 알림
"""
## ⚠️ 주의사항
- API 키는 절대 공개하지 마세요
- API 호출 제한: 일반 계정 하루 1,000회
- 데이터는 3시간마다 업데이트됩니다 (02, 05, 08, 11, 14, 17, 20, 23시)
- 과도한 API 호출 시 일시 차단될 수 있습니다

# 테스트

In [29]:
import requests
import json
from datetime import datetime

SERVICE_KEY = SERVICE_KEY

def test_api_connection():
    """API 연결 테스트"""
    print("🔍 API 연결 테스트 시작...")
    
    # 강남구 좌표로 테스트 (nx=61, ny=125)
    url = "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst"
    
    params = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": 10,
        "pageNo": 1,
        "dataType": "JSON",
        "base_date": datetime.now().strftime("%Y%m%d"),
        "base_time": "1400",  # 오후 2시
        "nx": 61,  # 강남구 x좌표
        "ny": 125  # 강남구 y좌표
    }
    
    try:
        print(f"📡 요청 URL: {url}")
        print(f"📋 요청 파라미터: {params}")
        
        response = requests.get(url, params=params)
        print(f"📊 응답 상태코드: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            print("✅ API 호출 성공!")
            
            # 응답 구조 확인
            if "response" in data:
                header = data["response"]["header"]
                print(f"🎯 결과코드: {header['resultCode']}")
                print(f"📝 결과메시지: {header['resultMsg']}")
                
                if header["resultCode"] == "00":
                    print("🎉 데이터 수신 성공!")
                    
                    # 받은 데이터 일부 출력
                    items = data["response"]["body"]["items"]["item"]
                    print(f"📈 수신된 데이터 개수: {len(items)}")
                    
                    print("\n🌡️ 강남구 현재 기상정보:")
                    for item in items[:5]:  # 처음 5개만 출력
                        category = item["category"]
                        value = item["obsrValue"]
                        
                        # 기상요소 한글 변환
                        weather_codes = {
                            "T1H": f"기온: {value}°C",
                            "RN1": f"1시간 강수량: {value}mm",
                            "UUU": f"동서바람성분: {value}m/s",
                            "VVV": f"남북바람성분: {value}m/s", 
                            "REH": f"습도: {value}%",
                            "PTY": f"강수형태: {value}",
                            "VEC": f"풍향: {value}도",
                            "WSD": f"풍속: {value}m/s"
                        }
                        
                        if category in weather_codes:
                            print(f"  • {weather_codes[category]}")
                    
                    return True
                else:
                    print(f"❌ API 오류: {header['resultMsg']}")
                    print("💡 가능한 원인:")
                    print("   - API 키가 잘못되었습니다")
                    print("   - API 사용 승인이 아직 완료되지 않았습니다")
                    print("   - 일일 사용량을 초과했습니다")
                    return False
            else:
                print("❌ 예상치 못한 응답 형식")
                print(json.dumps(data, indent=2, ensure_ascii=False))
                return False
        else:
            print(f"❌ HTTP 오류: {response.status_code}")
            return False
            
    except requests.exceptions.RequestException as e:
        print(f"❌ 네트워크 오류: {e}")
        return False
    except Exception as e:
        print(f"❌ 기타 오류: {e}")
        return False

In [30]:
def get_simple_district_weather(district_name, nx, ny):
    """특정 구의 간단한 날씨 정보 조회"""
    print(f"\n🏛️ {district_name} 날씨 조회 중...")
    
    url = "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtFcst"
    
    params = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": 60,
        "pageNo": 1,
        "dataType": "JSON",
        "base_date": datetime.now().strftime("%Y%m%d"),
        "base_time": "1400",
        "nx": nx,
        "ny": ny
    }
    
    try:
        response = requests.get(url, params=params)
        
        if response.status_code == 200:
            data = response.json()
            
            if data["response"]["header"]["resultCode"] == "00":
                items = data["response"]["body"]["items"]["item"]
                
                # 첫 번째 시간대 데이터만 추출
                first_time_data = {}
                for item in items:
                    if item["fcstTime"] == items[0]["fcstTime"]:  # 첫 번째 예보시간
                        first_time_data[item["category"]] = item["fcstValue"]
                
                print(f"⏰ 예보시간: {items[0]['fcstDate']} {items[0]['fcstTime']}")
                
                # 주요 정보 출력
                if "TMP" in first_time_data:
                    print(f"🌡️ 기온: {first_time_data['TMP']}°C")
                
                if "SKY" in first_time_data:
                    sky_code = int(first_time_data["SKY"])
                    sky_status = {1: "☀️ 맑음", 3: "⛅ 구름많음", 4: "☁️ 흐림"}
                    print(f"🌤️ 하늘상태: {sky_status.get(sky_code, '정보없음')}")
                
                if "PTY" in first_time_data:
                    pty_code = int(first_time_data["PTY"])
                    pty_status = {0: "☔ 강수없음", 1: "🌧️ 비", 2: "🌦️ 비/눈", 3: "❄️ 눈"}
                    print(f"🌦️ 강수형태: {pty_status.get(pty_code, '정보없음')}")
                
                if "POP" in first_time_data:
                    print(f"☔ 강수확률: {first_time_data['POP']}%")
                
                if "REH" in first_time_data:
                    print(f"💧 습도: {first_time_data['REH']}%")
                    
                return True
            else:
                print(f"❌ {district_name} 데이터 조회 실패")
                return False
        else:
            print(f"❌ HTTP 오류: {response.status_code}")
            return False
            
    except Exception as e:
        print(f"❌ 오류: {e}")
        return False

In [31]:
if __name__ == "__main__":
    print("=" * 50)
    print("🌤️ 기상청 API 테스트 프로그램")
    print("=" * 50)
    
    # 1. API 연결 테스트
    if test_api_connection():
        print("\n" + "=" * 50)
        
        # 2. 몇 개 구의 날씨 정보 조회
        test_districts = {
            "강남구": {"nx": 61, "ny": 125},
            "종로구": {"nx": 60, "ny": 127},
            "마포구": {"nx": 59, "ny": 126}
        }
        
        for district, coords in test_districts.items():
            get_simple_district_weather(district, coords["nx"], coords["ny"])
        
        print("\n🎉 테스트 완료!")
        print("\n💡 다음 단계:")
        print("   1. 위의 전체 가이드 코드를 사용해서 25개구 데이터 수집")
        print("   2. 데이터 분석 및 시각화")
        print("   3. 정기적 데이터 수집 시스템 구축")
        
    else:
        print("\n❌ API 연결에 실패했습니다.")
        print("\n🔧 문제 해결 방법:")
        print("   1. API 키가 정확한지 확인")
        print("   2. 공공데이터포털에서 API 승인 상태 확인")
        print("   3. 네트워크 연결 상태 확인")
        print("   4. API 사용량 제한 확인")

🌤️ 기상청 API 테스트 프로그램
🔍 API 연결 테스트 시작...
📡 요청 URL: http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst
📋 요청 파라미터: {'serviceKey': 'SERVICE_KEY', 'numOfRows': 10, 'pageNo': 1, 'dataType': 'JSON', 'base_date': '20250629', 'base_time': '1400', 'nx': 61, 'ny': 125}
📊 응답 상태코드: 200
❌ 네트워크 오류: Expecting value: line 1 column 1 (char 0)

❌ API 연결에 실패했습니다.

🔧 문제 해결 방법:
   1. API 키가 정확한지 확인
   2. 공공데이터포털에서 API 승인 상태 확인
   3. 네트워크 연결 상태 확인
   4. API 사용량 제한 확인


# 테스트2

In [34]:
# API 키 로드 디버깅 코드

import os
from dotenv import load_dotenv
import requests

# .env 파일 로드
print("🔧 .env 파일 로드 중...")
load_result = load_dotenv()
print(f"✅ .env 파일 로드 결과: {load_result}")

# 현재 작업 디렉토리 확인
print(f"📁 현재 작업 디렉토리: {os.getcwd()}")

# .env 파일 존재 확인
env_file_exists = os.path.exists('.env')
print(f"📄 .env 파일 존재 여부: {env_file_exists}")

# API 키 로드 확인
SERVICE_KEY = os.getenv('SERVICE_KEY')
print(f"🔑 로드된 API 키 길이: {len(SERVICE_KEY) if SERVICE_KEY else 0}")
print(f"🔑 API 키 앞 10자리: {SERVICE_KEY[:10] if SERVICE_KEY else 'None'}")
print(f"🔑 API 키 뒷 10자리: {SERVICE_KEY[-10:] if SERVICE_KEY else 'None'}")

# API 키가 제대로 로드되었는지 확인
if SERVICE_KEY is None:
    print("❌ API 키가 None입니다!")
    print("\n🔧 해결 방법:")
    print("1. .env 파일이 현재 디렉토리에 있는지 확인")
    print("2. .env 파일에서 SERVICE_KEY=실제키값 형태로 작성되었는지 확인")
    print("3. .env 파일에 공백이나 따옴표가 없는지 확인")
elif SERVICE_KEY == 'SERVICE_KEY':
    print("❌ API 키가 문자열 'SERVICE_KEY'로 설정되어 있습니다!")
elif len(SERVICE_KEY) < 50:
    print("⚠️ API 키가 너무 짧습니다. 올바른 키인지 확인해주세요.")
else:
    print("✅ API 키가 정상적으로 로드된 것 같습니다.")

# 환경변수 전체 확인 (민감정보 제외)
print(f"\n📋 현재 설정된 환경변수 개수: {len(os.environ)}")
service_key_in_env = 'SERVICE_KEY' in os.environ
print(f"🔍 SERVICE_KEY 환경변수 존재: {service_key_in_env}")

# .env 파일 내용 확인 (있다면)
if env_file_exists:
    print("\n📖 .env 파일 내용 확인:")
    try:
        with open('.env', 'r', encoding='utf-8') as f:
            lines = f.readlines()
        
        for i, line in enumerate(lines[:5]):  # 처음 5줄만
            if 'SERVICE_KEY' in line:
                # API 키 마스킹
                if '=' in line:
                    key, value = line.split('=', 1)
                    masked_value = value[:10] + '*' * (len(value.strip()) - 20) + value[-10:] if len(value.strip()) > 20 else '*' * len(value.strip())
                    print(f"  라인 {i+1}: {key}={masked_value}")
            else:
                print(f"  라인 {i+1}: {line.strip()}")
    except Exception as e:
        print(f"❌ .env 파일 읽기 오류: {e}")

print("\n" + "="*50)

🔧 .env 파일 로드 중...
✅ .env 파일 로드 결과: True
📁 현재 작업 디렉토리: C:\ai_x\ideanote\250617_1팀프로젝트\project01
📄 .env 파일 존재 여부: True
🔑 로드된 API 키 길이: 100
🔑 API 키 앞 10자리: Sg1QxDVRQ0
🔑 API 키 뒷 10자리: G2hQ%3D%3D
✅ API 키가 정상적으로 로드된 것 같습니다.

📋 현재 설정된 환경변수 개수: 67
🔍 SERVICE_KEY 환경변수 존재: True

📖 .env 파일 내용 확인:
  라인 1: SERVICE_KEY=Sg1QxDVRQ0********************************************************************************G2hQ%3D%3D



#확인

In [36]:
# 수정된 기상청 API 테스트 코드 (디버깅 강화)

import requests
import json
from datetime import datetime
import os
from dotenv import load_dotenv

# .env 파일 로드 및 확인
print("🔧 환경 설정 확인...")
load_dotenv()

SERVICE_KEY = os.getenv('SERVICE_KEY')

# API 키 확인
if not SERVICE_KEY:
    print("❌ SERVICE_KEY를 찾을 수 없습니다!")
    print("💡 .env 파일에 다음과 같이 작성했는지 확인하세요:")
    print("SERVICE_KEY=실제_API_키_값")
    exit()

print(f"✅ API 키 로드 완료 (길이: {len(SERVICE_KEY)})")

def test_api_connection_debug():
    """디버깅 강화된 API 연결 테스트"""
    print("\n🔍 API 연결 테스트 시작...")
    
    # 강남구 좌표로 테스트 (nx=61, ny=125)
    url = "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst"
    
    # 현재 시간 기반으로 적절한 base_time 계산
    now = datetime.now()
    current_hour = now.hour
    
    # 정시에서 40분 이후에 데이터가 제공되므로 이전 시간 사용
    if now.minute < 40:
        current_hour -= 1
    
    # 24시간 형식으로 변환
    if current_hour < 0:
        current_hour = 23
        base_date = (now.replace(hour=0, minute=0, second=0, microsecond=0) - 
                    datetime.timedelta(days=1)).strftime("%Y%m%d")
    else:
        base_date = now.strftime("%Y%m%d")
    
    base_time = f"{current_hour:02d}00"
    
    params = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": 10,
        "pageNo": 1,
        "dataType": "JSON",
        "base_date": base_date,
        "base_time": base_time,
        "nx": 61,  # 강남구 x좌표
        "ny": 125  # 강남구 y좌표
    }
    
    print(f"📡 요청 URL: {url}")
    print(f"📅 요청 날짜: {base_date}")
    print(f"⏰ 요청 시간: {base_time}")
    print(f"📍 좌표: nx={params['nx']}, ny={params['ny']}")
    print(f"🔑 API 키 앞 10자리: {SERVICE_KEY[:10]}...")
    
    try:
        response = requests.get(url, params=params, timeout=30)
        print(f"📊 응답 상태코드: {response.status_code}")
        print(f"📏 응답 크기: {len(response.text)} 문자")
        
        # 응답 내용 일부 출력 (디버깅용)
        print(f"📄 응답 내용 (처음 200자): {response.text[:200]}")
        
        if response.status_code == 200:
            # JSON 파싱 시도
            try:
                data = response.json()
                print("✅ JSON 파싱 성공!")
                
                # 응답 구조 확인
                if "response" in data:
                    header = data["response"]["header"]
                    print(f"🎯 결과코드: {header['resultCode']}")
                    print(f"📝 결과메시지: {header['resultMsg']}")
                    
                    if header["resultCode"] == "00":
                        print("🎉 데이터 수신 성공!")
                        
                        # 받은 데이터 일부 출력
                        body = data["response"]["body"]
                        if "items" in body and body["items"]:
                            items = body["items"]["item"]
                            print(f"📈 수신된 데이터 개수: {len(items)}")
                            
                            print("\n🌡️ 강남구 현재 기상정보:")
                            for item in items[:5]:  # 처음 5개만 출력
                                category = item["category"]
                                value = item["obsrValue"]
                                
                                # 기상요소 한글 변환
                                weather_codes = {
                                    "T1H": f"기온: {value}°C",
                                    "RN1": f"1시간 강수량: {value}mm",
                                    "UUU": f"동서바람성분: {value}m/s",
                                    "VVV": f"남북바람성분: {value}m/s", 
                                    "REH": f"습도: {value}%",
                                    "PTY": f"강수형태: {value}",
                                    "VEC": f"풍향: {value}도",
                                    "WSD": f"풍속: {value}m/s"
                                }
                                
                                if category in weather_codes:
                                    print(f"  • {weather_codes[category]}")
                            
                            return True
                        else:
                            print("❌ 응답에 데이터가 없습니다.")
                            print(f"📊 Body 내용: {body}")
                            return False
                    else:
                        print(f"❌ API 오류: {header['resultMsg']}")
                        print("\n💡 가능한 원인:")
                        if header["resultCode"] == "01":
                            print("   - 애플리케이션 에러")
                        elif header["resultCode"] == "02":
                            print("   - 데이터베이스 에러")
                        elif header["resultCode"] == "03":
                            print("   - 데이터없음 에러")
                        elif header["resultCode"] == "04":
                            print("   - HTTP 에러")
                        elif header["resultCode"] == "05":
                            print("   - 서비스 연결실패 에러")
                        elif header["resultCode"] == "20":
                            print("   - 서비스 접근거부")
                        elif header["resultCode"] == "21":
                            print("   - 일시적 서비스 장애")
                        elif header["resultCode"] == "30":
                            print("   - 등록되지 않은 서비스키")
                        elif header["resultCode"] == "31":
                            print("   - 기한만료된 서비스키")
                        elif header["resultCode"] == "32":
                            print("   - 등록되지 않은 IP")
                        elif header["resultCode"] == "33":
                            print("   - 서명되지 않은 호출")
                        else:
                            print(f"   - 알 수 없는 오류 코드: {header['resultCode']}")
                        return False
                else:
                    print("❌ 예상치 못한 응답 형식")
                    print("📄 전체 응답:")
                    print(json.dumps(data, indent=2, ensure_ascii=False))
                    return False
                    
            except json.JSONDecodeError as e:
                print(f"❌ JSON 파싱 오류: {e}")
                print("📄 응답이 JSON 형식이 아닙니다:")
                print(response.text)
                return False
                
        else:
            print(f"❌ HTTP 오류: {response.status_code}")
            print(f"📄 응답 내용: {response.text}")
            return False
            
    except requests.exceptions.Timeout:
        print("❌ 요청 시간 초과 (30초)")
        return False
    except requests.exceptions.RequestException as e:
        print(f"❌ 네트워크 오류: {e}")
        return False
    except Exception as e:
        print(f"❌ 기타 오류: {e}")
        return False

# 🧪 테스트 실행
if __name__ == "__main__":
    print("=" * 60)
    print("🌤️ 기상청 API 디버깅 테스트 프로그램")
    print("=" * 60)
    
    # API 연결 테스트
    success = test_api_connection_debug()
    
    if success:
        print("\n🎉 API 연결 성공!")
    else:
        print("\n❌ API 연결 실패")
        print("\n🔧 문제 해결 체크리스트:")
        print("   ✓ API 키가 올바른지 확인")
        print("   ✓ 공공데이터포털에서 서비스 승인 상태 확인")
        print("   ✓ API 사용량 제한 확인 (일반계정: 1,000회/일)")
        print("   ✓ 네트워크 방화벽 설정 확인")

🔧 환경 설정 확인...
✅ API 키 로드 완료 (길이: 100)
🌤️ 기상청 API 디버깅 테스트 프로그램

🔍 API 연결 테스트 시작...
📡 요청 URL: http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst
📅 요청 날짜: 20250629
⏰ 요청 시간: 2000
📍 좌표: nx=61, ny=125
🔑 API 키 앞 10자리: Sg1QxDVRQ0...
📊 응답 상태코드: 200
📏 응답 크기: 229 문자
📄 응답 내용 (처음 200자): <OpenAPI_ServiceResponse>
	<cmmMsgHeader>
		<errMsg>SERVICE ERROR</errMsg>
		<returnAuthMsg>SERVICE_KEY_IS_NOT_REGISTERED_ERROR</returnAuthMsg>
		<returnReasonCode>30</returnReasonCode>
	</cmmMsgHeade
❌ JSON 파싱 오류: Expecting value: line 1 column 1 (char 0)
📄 응답이 JSON 형식이 아닙니다:
<OpenAPI_ServiceResponse>
	<cmmMsgHeader>
		<errMsg>SERVICE ERROR</errMsg>
		<returnAuthMsg>SERVICE_KEY_IS_NOT_REGISTERED_ERROR</returnAuthMsg>
		<returnReasonCode>30</returnReasonCode>
	</cmmMsgHeader>
</OpenAPI_ServiceResponse>

❌ API 연결 실패

🔧 문제 해결 체크리스트:
   ✓ API 키가 올바른지 확인
   ✓ 공공데이터포털에서 서비스 승인 상태 확인
   ✓ API 사용량 제한 확인 (일반계정: 1,000회/일)
   ✓ 네트워크 방화벽 설정 확인


# 테스트3 api

In [1]:
# # 승인된 API인데도 오류가 날 때 상세 디버깅

# import requests
# import json
# from datetime import datetime, timedelta
# import os
# from dotenv import load_dotenv
# import urllib.parse

# # .env 파일 로드
# load_dotenv()
# SERVICE_KEY = os.getenv('SERVICE_KEY')

# def debug_service_key_issues():
#     """서비스키 관련 상세 디버깅"""
#     print("🔍 서비스키 상세 분석...")
    
#     if not SERVICE_KEY:
#         print("❌ SERVICE_KEY가 없습니다!")
#         return False
    
#     print(f"📏 키 길이: {len(SERVICE_KEY)}")
#     print(f"🔑 키 시작: {SERVICE_KEY[:20]}...")
#     print(f"🔑 키 끝: ...{SERVICE_KEY[-20:]}")
    
#     # 키에 특수문자나 공백 확인
#     has_space = ' ' in SERVICE_KEY
#     has_newline = '\n' in SERVICE_KEY or '\r' in SERVICE_KEY
#     has_tab = '\t' in SERVICE_KEY
    
#     print(f"🔍 공백 포함: {has_space}")
#     print(f"🔍 개행 포함: {has_newline}")
#     print(f"🔍 탭 포함: {has_tab}")
    
#     if has_space or has_newline or has_tab:
#         print("⚠️ 키에 공백/개행/탭이 포함되어 있습니다!")
#         cleaned_key = SERVICE_KEY.strip().replace('\n', '').replace('\r', '').replace('\t', '')
#         print(f"🧹 정리된 키: {cleaned_key}")
#         return cleaned_key
    
#     return SERVICE_KEY

# def test_different_apis():
#     """다른 기상청 API들도 테스트해보기"""
    
#     # 정리된 키 사용
#     clean_key = debug_service_key_issues()
    
#     # 여러 기상청 API 엔드포인트들
#     api_endpoints = {
#         "단기예보_초단기실황": "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst",
#         "단기예보_초단기예보": "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtFcst",
#         "단기예보_단기예보": "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst",
#         "기상특보": "http://apis.data.go.kr/1360000/WthrWrnInfoService/getWthrWrnList",
#         "지상시간자료": "http://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList"
#     }
    
#     # 시간 설정 (좀 더 과거 시간으로)
#     now = datetime.now()
#     base_date = (now - timedelta(hours=3)).strftime("%Y%m%d")
#     base_time = "1400"  # 오후 2시 고정
    
#     for api_name, url in api_endpoints.items():
#         print(f"\n{'='*60}")
#         print(f"🧪 {api_name} API 테스트...")
#         print(f"🔗 URL: {url}")
        
#         # API별로 다른 파라미터 설정
#         if "WthrWrnInfoService" in url:
#             # 기상특보는 파라미터가 다름
#             params = {
#                 "serviceKey": clean_key,
#                 "numOfRows": 10,
#                 "pageNo": 1,
#                 "dataType": "JSON",
#                 "fromTmFc": base_date,
#                 "toTmFc": base_date
#             }
#         elif "AsosHourlyInfoService" in url:
#             # 지상시간자료는 파라미터가 다름
#             params = {
#                 "serviceKey": clean_key,
#                 "numOfRows": 10,
#                 "pageNo": 1,
#                 "dataType": "JSON",
#                 "dataCd": "ASOS",
#                 "dateCd": "HR",
#                 "startDt": base_date,
#                 "startHh": "14",
#                 "endDt": base_date,
#                 "endHh": "14",
#                 "stnIds": "108"  # 서울 관측소
#             }
#         else:
#             # 단기예보 계열
#             params = {
#                 "serviceKey": clean_key,
#                 "numOfRows": 10,
#                 "pageNo": 1,
#                 "dataType": "JSON",
#                 "base_date": base_date,
#                 "base_time": base_time,
#                 "nx": 61,
#                 "ny": 125
#             }
        
#         try:
#             print(f"📋 요청 파라미터: {dict(list(params.items())[:3])}...")  # 처음 3개만 출력
            
#             response = requests.get(url, params=params, timeout=15)
#             print(f"📊 응답 상태코드: {response.status_code}")
#             print(f"📏 응답 크기: {len(response.text)} 문자")
            
#             if response.status_code == 200:
#                 response_text = response.text
                
#                 # XML 응답 체크
#                 if response_text.strip().startswith('<'):
#                     print("📄 XML 응답:")
#                     print(response_text[:300])
                    
#                     if "NORMAL_SERVICE" in response_text:
#                         print("🎉 성공! 이 API는 정상 작동합니다!")
#                         return api_name, url, clean_key
#                     elif "SERVICE_KEY_IS_NOT_REGISTERED_ERROR" in response_text:
#                         print("❌ 등록되지 않은 서비스키 오류")
#                     elif "SERVICE_ACCESS_DENIED_ERROR" in response_text:
#                         print("❌ 서비스 접근 거부 오류")
#                     elif "SERVICE_TEMPORARILY_OUT_OF_ORDER_ERROR" in response_text:
#                         print("❌ 일시적 서비스 장애")
#                     else:
#                         print("❓ 기타 XML 응답")
                        
#                 # JSON 응답 체크
#                 else:
#                     try:
#                         data = response.json()
#                         header = data.get("response", {}).get("header", {})
#                         result_code = header.get("resultCode")
#                         result_msg = header.get("resultMsg")
                        
#                         print(f"🎯 결과코드: {result_code}")
#                         print(f"📝 결과메시지: {result_msg}")
                        
#                         if result_code == "00":
#                             print("🎉 성공! 이 API는 정상 작동합니다!")
#                             return api_name, url, clean_key
#                         else:
#                             print(f"❌ API 오류: {result_msg}")
#                     except:
#                         print("❌ JSON 파싱 실패, 응답 내용:")
#                         print(response_text[:200])
#             else:
#                 print(f"❌ HTTP 오류: {response.status_code}")
                
#         except Exception as e:
#             print(f"❌ 요청 오류: {e}")
        
#         print("⏳ 3초 대기...")
#         import time
#         time.sleep(3)  # API 호출 간격
    
#     return None, None, None

# def check_api_approval_details():
#     """API 승인 상세 확인 가이드"""
#     print(f"\n{'='*60}")
#     print("📋 API 승인 상세 확인 방법:")
#     print("\n1. 공공데이터포털 마이페이지에서 확인할 것들:")
#     print("   🔍 오픈API → 개발계정에서 각 API별 상태")
#     print("   🔍 서비스키가 여러 개인지 확인")
#     print("   🔍 트래픽 제한 확인 (기본: 1,000회/일)")
    
#     print("\n2. 각 API별 개별 승인 여부:")
#     api_list = [
#         "기상청_수치모델자료(경량화) 조회서비스",
#         "기상청_기상특보 조회서비스", 
#         "기상청_레이더영상 조회서비스",
#         "기상청_지상(종관,ASOS) 일자료 조회서비스",
#         "기상청_지상(종관,ASOS) 시간자료 조회서비스",
#         "기상청_단기예보((구)_동네예보) 조회서비스"
#     ]
    
#     for i, api in enumerate(api_list, 1):
#         print(f"   {i}. {api}")
    
#     print("\n3. 가능한 문제들:")
#     print("   ⚠️ 일부 API만 승인되고 일부는 승인대기")
#     print("   ⚠️ 승인 후 시스템 반영 지연 (최대 1-2시간)")
#     print("   ⚠️ 서비스키 복사시 공백이나 특수문자 포함")
#     print("   ⚠️ 잘못된 서비스키 사용 (여러 개 중 다른 키)")

# def try_manual_key_input():
#     """수동으로 키를 입력받아 테스트"""
#     print(f"\n{'='*60}")
#     print("🔧 수동 키 입력 테스트")
#     print("📝 공공데이터포털에서 서비스키를 다시 복사해서 붙여넣어 주세요:")
#     print("   (마이페이지 → 오픈API → 개발계정 → 인증키 확인)")
    
#     manual_key = input("\n🔑 서비스키 입력: ").strip()
    
#     if manual_key:
#         print(f"✅ 입력된 키 길이: {len(manual_key)}")
#         print(f"🔑 키 시작: {manual_key[:20]}...")
        
#         # 간단한 테스트
#         url = "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst"
#         params = {
#             "serviceKey": manual_key,
#             "numOfRows": 5,
#             "pageNo": 1,
#             "dataType": "JSON",
#             "base_date": "20250629",
#             "base_time": "1400",
#             "nx": 61,
#             "ny": 125
#         }
        
#         try:
#             response = requests.get(url, params=params, timeout=10)
#             print(f"📊 응답 상태코드: {response.status_code}")
            
#             if "NORMAL_SERVICE" in response.text:
#                 print("🎉 성공! 이 키가 올바른 키입니다!")
#                 print(f"📝 .env 파일을 다음으로 수정하세요:")
#                 print(f"SERVICE_KEY={manual_key}")
#                 return True
#             else:
#                 print("❌ 여전히 오류 발생")
#                 print(response.text[:200])
#         except Exception as e:
#             print(f"❌ 테스트 오류: {e}")
    
#     return False

# # 🧪 메인 실행
# if __name__ == "__main__":
#     print("🔍 승인된 API 오류 상세 진단 시작...")
    
#     # 1. 여러 API 테스트
#     working_api, working_url, working_key = test_different_apis()
    
#     if working_api:
#         print(f"\n🎉 성공! '{working_api}' API가 정상 작동합니다!")
#         print(f"🔗 성공한 URL: {working_url}")
#         print(f"🔑 성공한 키: {working_key[:20]}...")
#     else:
#         print("\n❌ 모든 API 테스트 실패")
#         check_api_approval_details()
        
#         # 2. 수동 키 입력 테스트
#         print("\n" + "="*60)
#         if try_manual_key_input():
#             print("✅ 수동 입력으로 해결됨!")
#         else:
#             print("\n🤔 추가 확인이 필요합니다.")
#             print("💬 다음 정보를 확인해 주세요:")
#             print("   1. 마이페이지에서 서비스키가 몇 개인지")
#             print("   2. 각 API별 승인 상태 스크린샷")
#             print("   3. 승인 완료 시간 (언제 승인되었는지)")